# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Print dataset name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we enumerate the record sets and their respective fields using their `@id` values.


In [ ]:
# List all record sets and their fields by @id
record_sets = []
# `dataset.metadata.record_sets` may not be present - fallback to searching for them
if hasattr(dataset.metadata, 'record_sets') and dataset.metadata.record_sets:
    record_sets = dataset.metadata.record_sets
else:
    # Try to find all record sets in the metadata object
    # Extract from the top-level metadata
    for attr in dir(dataset.metadata):
        value = getattr(dataset.metadata, attr)
        if isinstance(value, list):
            for item in value:
                if hasattr(item, '@type') and item['@type'] == 'RecordSet':
                    record_sets.append(item)
        elif isinstance(value, dict):
            if value.get('@type', None) == 'RecordSet':
                record_sets.append(value)

if not record_sets:
    # Try to enumerate from .record_sets()
    try:
        record_sets = list(dataset.record_sets())
    except Exception as e:
        print('Could not enumerate record sets:', e)

if not record_sets:
    print('No record sets found in this schema!')
else:
    print(f"Record Sets found ({len(record_sets)}):\n")
    for rs in record_sets:
        rs_id = getattr(rs, '@id', None) if hasattr(rs, '@id') else rs.get('@id', None)
        rs_name = getattr(rs, 'name', None) if hasattr(rs, 'name') else rs.get('name', None)
        print(f"- RecordSet @id: {rs_id}  |  name: {rs_name}")
        # List fields
        if hasattr(rs, 'fields') and rs.fields:
            for field in rs.fields:
                field_id = getattr(field, '@id', None) if hasattr(field, '@id') else field.get('@id', None)
                field_name = getattr(field, 'name', None) if hasattr(field, 'name') else field.get('name', None)
                print(f"    - Field @id: {field_id}  |  name: {field_name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we will attempt to programmatically extract all available record sets (using their `@id`s) and load them into Pandas DataFrames.


In [ ]:
# A helper to extract record set @id strings (or use from the overview)
def get_record_set_ids(ds):
    ids = []
    # Try to use .record_sets() generator
    try:
        for rs in ds.record_sets():
            rs_id = getattr(rs, '@id', None) if hasattr(rs, '@id') else rs.get('@id', None)
            ids.append(rs_id)
    except Exception:
        pass
    return ids

record_set_ids = get_record_set_ids(dataset)
if not record_set_ids:
    # Try a common default for mlcroissant packaging, fallback to 'cr:MainTable'
    record_set_ids = ['cr:MainTable']

dataframes = dict()
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"✔ Loaded record set {record_set_id}: {df.shape[0]} rows, {df.shape[1]} columns")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# For demonstration, pick the first loaded DataFrame
example_record_set_id = list(dataframes.keys())[0] if dataframes else None

if example_record_set_id:
    print(f"\nColumns in record set '{example_record_set_id}':")
    print(dataframes[example_record_set_id].columns.tolist())
    dataframes[example_record_set_id].head()
else:
    print('No dataframes loaded!')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Below we select and process numeric and categorical fields by their `@id` or column name.

In [ ]:
import numpy as np

# For demonstration, select a numeric field from the available columns
df = dataframes[example_record_set_id]

print('First five rows:')
display(df.head())

# Find candidate numeric fields (float or int columns)
numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_candidates:
    print("No numeric fields found. Trying to coerce columns to numeric...")
    # Try to coerce all columns to numeric where possible
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_candidates:
    raise ValueError('No numeric columns found to analyze!')
numeric_field = numeric_candidates[0]
print(f"Selected numeric field: {numeric_field}")

threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}: {filtered_df.shape[0]} rows")
display(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a likely categorical field
category_candidates = df.select_dtypes(include=[object]).columns.tolist()
group_field = None
for c in category_candidates:
    if 'type' in c.lower() or 'class' in c.lower() or 'group' in c.lower() or 'category' in c.lower() or 'sample' in c.lower():
        group_field = c
        break
if not group_field and category_candidates:
    group_field = category_candidates[0]

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
    print(f"Grouped data by '{group_field}':")
    display(grouped_df.head())
else:
    print('No suitable group field found for grouping analysis.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# If we grouped by a field, show mean value per group
if group_field:
    plt.figure(figsize=(10, 4))
    sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field])
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xticks(rotation=60, ha='right')
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, inspect, process, and visualize the Fair^2 dataset: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells, using the `mlcroissant` library.

- The dataset was loaded via its Croissant JSON-LD schema using its URL.
- We inspected available record sets, fields, and their `@id`s for precise referencing.
- We extracted tabular data and performed basic filtering, normalization, and grouping.
- Distributions and group means were visualized for a selected numeric field.

You may tailor further analyses to your scientific use case. For more information, consult the [mlcroissant documentation](https://github.com/mlcommons/croissant).
